# Neural Network from Scratch — Analysis

Trains a fully-connected network on MNIST using only NumPy, then visualizes:
- Training curves (loss + accuracy)
- Learned weight filters
- Misclassified examples
- Confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from neural_network import init_params, init_adam_state, forward_pass, backward_pass, adam_update, accuracy, cross_entropy_loss, predict
from utils import load_mnist, one_hot_encode, get_mini_batches

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
X_train, X_test, Y_train, Y_test = load_mnist()
print(f"Train: {X_train.shape[1]:,} samples | Test: {X_test.shape[1]:,} samples")

## 2. Train

In [ ]:
EPOCHS      = 20
BATCH_SIZE  = 64
HIDDEN_SIZE = 128
LR          = 0.001

Y_train_oh = one_hot_encode(Y_train)
params     = init_params(hidden_size=HIDDEN_SIZE)
adam_state = init_adam_state(params)
rng        = np.random.default_rng(0)

history = {"train_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    epoch_loss, num_batches = 0.0, 0
    for X_batch, Y_batch in get_mini_batches(X_train, Y_train_oh, BATCH_SIZE, rng):
        A2, cache  = forward_pass(X_batch, params)
        epoch_loss += cross_entropy_loss(A2, Y_batch)
        grads      = backward_pass(X_batch, Y_batch, params, cache)
        params, adam_state = adam_update(params, grads, adam_state, lr=LR)
        num_batches += 1

    avg_loss  = epoch_loss / num_batches
    train_acc = accuracy(X_train, Y_train, params)
    test_acc  = accuracy(X_test,  Y_test,  params)

    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch:>2}/{EPOCHS}  loss: {avg_loss:.4f}  train: {train_acc:.4f}  test: {test_acc:.4f}")

print(f"\nFinal test accuracy: {history['test_acc'][-1]*100:.2f}%")

## 3. Training Curves

In [ ]:
epochs = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], color="steelblue")
ax1.set_title("Cross-Entropy Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(alpha=0.3)

ax2.plot(epochs, [a * 100 for a in history["train_acc"]], label="Train", color="steelblue")
ax2.plot(epochs, [a * 100 for a in history["test_acc"]],  label="Test",  color="tomato")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Learned Weight Filters

Each of the 128 hidden neurons has a 784-dimensional weight vector. Reshaped to 28×28, these reveal what visual pattern each neuron responds to.

In [ ]:
W1 = params["W1"]  # (128, 784)

fig, axes = plt.subplots(8, 16, figsize=(16, 8))
fig.suptitle("Learned W1 Filters (each neuron's 28×28 weight pattern)", fontsize=13)

for i, ax in enumerate(axes.flat):
    w = W1[i].reshape(28, 28)
    vmax = np.abs(w).max()
    ax.imshow(w, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Misclassified Examples

In [ ]:
preds = predict(X_test, params)
wrong_idx = np.where(preds != Y_test)[0]
print(f"Misclassified: {len(wrong_idx)} / {len(Y_test)} ({len(wrong_idx)/len(Y_test)*100:.1f}%)")

# Show first 40 mistakes
show = wrong_idx[:40]
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
fig.suptitle("Misclassified Test Images  (true → predicted)", fontsize=13)

for ax, idx in zip(axes.flat, show):
    ax.imshow(X_test[:, idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"{Y_test[idx]}→{preds[idx]}", fontsize=8, color="tomato")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Confusion Matrix

In [ ]:
cm = np.zeros((10, 10), dtype=int)
for true, pred in zip(Y_test, preds):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=13)

for i in range(10):
    for j in range(10):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9, color=color)

plt.tight_layout()
plt.show()